In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel('../data/E Commerce Dataset.xlsx', sheet_name='E Comm')

# Повторюємо очистку
num_cols = ['DaySinceLastOrder', 'OrderAmountHikeFromlastYear', 'Tenure',
            'OrderCount', 'CouponUsed', 'HourSpendOnApp', 'WarehouseToHome']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [2]:
df['Recency'] = df['DaySinceLastOrder']
df['Frequency'] = df['OrderCount']
df['Monetary'] = df['CashbackAmount']

# RFM score — чим менший recency і більший frequency/monetary, тим краще
df['RFM_Score'] = (
    df['Frequency'] * 0.4 +
    df['Monetary'] / 100 * 0.4 +
    (1 / (df['Recency'] + 1)) * 0.2
).round(3)

In [3]:
# Юзер новий якщо tenure менше 3 місяців
df['IsNewUser'] = (df['Tenure'] < 3).astype(int)

# Активність на платформі
df['EngagementScore'] = (
    df['HourSpendOnApp'] * 0.5 +
    df['NumberOfDeviceRegistered'] * 0.3 +
    df['CouponUsed'] * 0.2
).round(3)

# Скарга + низька задоволеність = сильний сигнал
df['ComplainLowSat'] = (
    (df['Complain'] == 1) & (df['SatisfactionScore'] <= 2)
).astype(int)

In [4]:
cat_cols = ['PreferredLoginDevice', 'PreferredPaymentMode', 
            'PreferedOrderCat', 'MaritalStatus', 'Gender']

df_model = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Прибираємо CustomerID — він не потрібен моделі
df_model = df_model.drop(columns=['CustomerID'])

print(df_model.shape)
df_model.head()

(5630, 37)


,Churn,Tenure,CityTier,WarehouseToHome,HourSpendOnApp,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,...,PreferredPaymentMode_E wallet,PreferredPaymentMode_UPI,PreferedOrderCat_Grocery,PreferedOrderCat_Laptop & Accessory,PreferedOrderCat_Mobile,PreferedOrderCat_Mobile Phone,PreferedOrderCat_Others,MaritalStatus_Married,MaritalStatus_Single,Gender_Male
0,1,4.0,3,6.0,3.0,3,2,9,1,11.0,...,False,False,False,True,False,False,False,False,True,False
1,1,9.0,1,8.0,3.0,4,3,7,1,15.0,...,False,True,False,False,True,False,False,False,True,True
2,1,9.0,1,30.0,2.0,4,3,6,1,14.0,...,False,False,False,False,True,False,False,False,True,True
3,1,0.0,3,15.0,2.0,4,5,8,0,23.0,...,False,False,False,True,False,False,False,False,True,True
4,1,0.0,1,12.0,3.0,3,5,3,0,11.0,...,False,False,False,False,True,False,False,False,True,True


In [6]:
# Конвертуємо булеві колонки в int
bool_cols = df_model.select_dtypes(include='bool').columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

print(df_model.shape)
df_model.dtypes.value_counts()

(5630, 37)


int64      24
float64    13
Name: count, dtype: int64

In [8]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Churn rate train: {y_train.mean().round(3)}')
print(f'Churn rate test: {y_test.mean().round(3)}')

Train: (4504, 36), Test: (1126, 36)
Churn rate train: 0.168
Churn rate test: 0.169


In [9]:
df_model.to_csv('../data/df_model.csv', index=False)
print("Збережено: data/df_model.csv")

Збережено: data/df_model.csv
